# TD/TP 3 — Unsupervised learning: PCA, autoencoders, and latent space

**IMT · Introduction to Machine Learning · Google Colab**    
**Based on course material:** Part 1: unsupervised learning/dimensionality reduction; Part 3: autoencoder encoder/bottleneck/decoder and interpolation; Part 4: generative design motivation

**Learning objectives:** reduce dimensionality without labels; train a two-dimensional autoencoder; visualize reconstructions and latent interpolations; cluster latent representations.

### How to use this notebook
1. Read the questions, run the prepared cells in order, and complete every **✍️ TODO** cell yourself.
2. The data preparation and plotting helpers are supplied so you can focus on machine learning.
3. If you get stuck, discuss your approach with a classmate before consulting the instructor.
4. You will receive the complete solution notebook after the collective correction.

**Environment:** Google Colab → Runtime → Run all

## Scenario: compressing images without labels

Again we use small handwritten digit images (8 × 8 = **64 pixel features**), but **we deliberately do not use digit labels to train**. The autoencoder must learn to reconstruct its input from a 2-dimensional bottleneck.

We compare a *linear* reduction (**PCA**) with a *nonlinear* autoencoder. Ground-truth digit labels are held aside only for evaluating/visualizing the learned representation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

SEED = 21
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
digits = load_digits()
X = digits.images.astype('float32').reshape(-1,64) / 16.0
y = digits.target  # For DISPLAY only, never for training the PCA or autoencoder
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.25, random_state=SEED, stratify=y
)
xtr = torch.tensor(X_train, dtype=torch.float32)
xte = torch.tensor(X_test, dtype=torch.float32)
loader = DataLoader(TensorDataset(xtr), batch_size=64, shuffle=True)

fig, axes = plt.subplots(1, 7, figsize=(9,2))
for i,ax in enumerate(axes):
    ax.imshow(X_test[i].reshape(8,8),cmap='gray',vmin=0,vmax=1); ax.axis('off')
plt.suptitle('Examples to compress; 64 pixels each')
plt.show()
print('Unlabeled training inputs:', X_train.shape)

## 1. ✍️ TODO — PCA as a first 2D representation

Fit `PCA(n_components=2)` on training images **without using labels**. Encode the test set, reconstruct it using `inverse_transform`, and measure the **mean squared reconstruction error**.

**Checkpoint:** latent coordinates have shape `(n_test, 2)`.

In [ ]:
# TODO: fit `pca`, create `z_pca` and `recon_pca`, compute `pca_mse`.
raise NotImplementedError('Fit PCA and reconstruct the test data')

assert z_pca.shape == (len(X_test), 2)
assert recon_pca.shape == X_test.shape
print(f'PCA 2D test reconstruction MSE: {pca_mse:.4f}')

## 2. ✍️ TODO — build an encoder → bottleneck → decoder

Implement an `Autoencoder` with:

- Encoder: `Linear(64, 32) → ReLU → Linear(32, 2)`
- Decoder: `Linear(2, 32) → ReLU → Linear(32, 64) → Sigmoid`
- `forward(x)` should return `decoder(encoder(x))`.

Why a **sigmoid** on the decoder's last layer? Because input pixel values are normalized into [0,1].

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: implement `self.encoder` and `self.decoder`.
        raise NotImplementedError('Build encoder and decoder')

    def forward(self, x):
        # TODO: encode then decode
        raise NotImplementedError('Connect the encoder and decoder')

model = Autoencoder()
assert model(xtr[:4]).shape == (4,64)
assert model.encoder(xtr[:4]).shape == (4,2)

## 3. ✍️ TODO — train using reconstruction error

Unsupervised **does not mean no loss function**. The target is simply the input image itself. Use `nn.MSELoss()`, `Adam(lr=0.01)`, and implement the update steps. Do **not** feed in digit labels.

In [ ]:
# TODO: set `criterion` (reconstruction loss) and `optimizer`.
raise NotImplementedError('Choose autoencoder training components')

def train_epoch(model, loader, optimizer):
    model.train()
    total = 0.0
    for (xb,) in loader:
        # TODO: forward -> reconstruction loss with target xb -> backward -> step
        raise NotImplementedError('Train an autoencoder minibatch')
        total += loss.item() * len(xb)
    return total/len(loader.dataset)

history = [train_epoch(model, loader, optimizer) for epoch in range(90)]
plt.plot(history); plt.xlabel('Epoch'); plt.ylabel('Reconstruction MSE'); plt.show()

## 4. ✍️ TODO — compare PCA and nonlinear reconstruction

Encode and reconstruct **unseen test images** with your autoencoder. Compare its MSE against PCA and display **original / PCA / autoencoder** versions of a few digits. Neither method has access to the digit labels during training.

**Question:** Does 2D provide enough information to reconstruct every digit perfectly?

In [ ]:
model.eval()
with torch.no_grad():
    # TODO: calculate `z_ae` and `recon_ae`, both as NumPy arrays.
    raise NotImplementedError('Encode and reconstruct the test data')

ae_mse = np.mean((recon_ae - X_test)**2)
print(f'PCA MSE: {pca_mse:.4f} | Autoencoder MSE: {ae_mse:.4f}')
fig, axes = plt.subplots(3, 7, figsize=(10,4))
for i in range(7):
    for row, item in enumerate([X_test[i], recon_pca[i], recon_ae[i]]):
        axes[row,i].imshow(item.reshape(8,8),cmap='gray',vmin=0,vmax=1)
        axes[row,i].axis('off')
for a,label in zip(axes[:,0],['Original','PCA','Autoencoder']):
    a.set_ylabel(label)
plt.tight_layout(); plt.show()

## 5. Latent space visualization (provided)

Use withheld test labels **only to color the scatterplot**, not for model fitting. The encoder has never been told the classes.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
for ax,z,title in zip(axes,[z_pca,z_ae],['PCA: 2D codes','Autoencoder: 2D codes']):
    scatter=ax.scatter(z[:,0],z[:,1],c=y_test,cmap='tab10',s=12,alpha=.75)
    ax.set(title=title,xlabel='Latent dimension 1',ylabel='Latent dimension 2')
fig.colorbar(scatter,ax=axes.ravel().tolist(),label='True digit (evaluation only)',shrink=.75)
plt.show()

## 6. ✍️ TODO — interpolate between two images in latent space

Select one test image of digit `1` and another of digit `7` (labels are used **only for selecting display examples**). Encode both, linearly interpolate 7 latent vectors, and decode each.

Formula: `z(alpha) = (1-alpha) * z_A + alpha * z_B` with `alpha` between 0 and 1.

This illustrates the latent interpolation idea discussed in the course; a regular autoencoder does **not** guarantee that all intermediate points decode realistically.

In [ ]:
id_a = np.flatnonzero(y_test == 1)[0]
id_b = np.flatnonzero(y_test == 7)[0]
with torch.no_grad():
    z_a = model.encoder(xte[id_a:id_a+1])
    z_b = model.encoder(xte[id_b:id_b+1])
    # TODO: construct `codes` of shape (7,2) by interpolating z_a to z_b.
    # Then calculate `images` with the decoder (NumPy array of shape (7,64)).
    raise NotImplementedError('Interpolate in 2D latent space')

assert images.shape == (7,64)
fig, axes = plt.subplots(1,7,figsize=(10,2))
for i,ax in enumerate(axes):
    ax.imshow(images[i].reshape(8,8),cmap='gray',vmin=0,vmax=1)
    ax.set(title=f'{i/6:.2f}',xticks=[],yticks=[])
plt.suptitle('Decoder output: 1 → 7 (latent interpolation)')
plt.show()

## 7. ✍️ TODO — clustering latent representations

Fit `KMeans(n_clusters=10, random_state=SEED, n_init=10)` on **training** latent codes and predict cluster IDs for test codes. Use **adjusted Rand index** to compare clusters with test labels (evaluation only). Cluster IDs are arbitrary and need not match digit numbers.

**Question:** Why might reconstruction features be unsuitable for perfect class separation?

In [ ]:
with torch.no_grad():
    z_train = model.encoder(xtr).numpy()
# TODO: fit kmeans on z_train and get `clusters_test` by predicting on z_ae.
raise NotImplementedError('Fit KMeans in latent space')

ari = adjusted_rand_score(y_test, clusters_test)
print(f'Adjusted Rand index (test set): {ari:.3f}')

### Discussion / optional challenge

1. How does a 2D bottleneck change reconstruction quality compared with 8 or 16 dimensions?
2. Why are the input and the target **identical** during autoencoder training?
3. Can we sample arbitrary random points in latent space and expect convincing digits?
4. **Challenge:** train another autoencoder with latent dimension 8 and compare MSE. How would you display this latent space?

**Takeaway:** Reconstruction is an unsupervised training signal; the learned compressed latent space can support visualization, interpolation, and downstream tasks.